# 04 — Gold Star Schema

Customer-facing star schema built on top of silver. Five dimensions and four
fact tables, plus three pre-aggregated views that drive the dashboard and
Genie space.

```
                    dim_date
                       │
   dim_team ───── fact_player_game_metrics ─── dim_player
       │                       │
       │            fact_player_shifts
       │                       │
       └───── fact_game_events ─── dim_player
              │
          dim_venue
```

| Gold table | Grain | Clustered by |
|------------|-------|--------------|
| `dim_team` | team | — |
| `dim_player` | player | — |
| `dim_venue` | venue | — |
| `dim_game` | game | `(season, game_date)` |
| `dim_date` | date | — |
| `fact_game_events` | event | `(season, game_date, team_id)` |
| `fact_player_shifts` | shift | `(game_id, player_id)` |
| `fact_player_game_metrics` | game × player × metric | `(game_id, player_id)` |
| `fact_player_season_metrics` | season × player × metric | `(season, player_id)` |

Three views:
- `v_team_standings`
- `v_player_season_leaders`
- `v_shot_map`

In [1]:
import os, json
from dotenv import load_dotenv
load_dotenv()

# Dual-mode Spark session — works locally via Databricks Connect AND inside a
# Databricks workspace notebook. In the workspace, `spark` already exists.
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "sportlogiq_nhl")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

print(f"Catalog: {UC_CATALOG}")
print(f"Bronze:  {UC_CATALOG}.{BRONZE_SCHEMA}")
print(f"Silver:  {UC_CATALOG}.{SILVER_SCHEMA}")
print(f"Gold:    {UC_CATALOG}.{GOLD_SCHEMA}")
print(f"Volume:  {VOLUME_PATH}")

Catalog: alexander_booth
Bronze:  alexander_booth.sportlogiq_nhl_bronze
Silver:  alexander_booth.sportlogiq_nhl_silver
Gold:    alexander_booth.sportlogiq_nhl_gold
Volume:  /Volumes/alexander_booth/sportlogiq_nhl_bronze/raw_data


In [3]:
S = f"{UC_CATALOG}.{SILVER_SCHEMA}"
G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
print(f"Silver: {S}\nGold:   {G}")

Silver: alexander_booth.sportlogiq_nhl_silver
Gold:   alexander_booth.sportlogiq_nhl_gold


## Dimensions

In [4]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_team AS
    SELECT
        MD5(CAST(team_id AS STRING)) AS team_sk,
        team_id,
        team_name,
        team_location,
        team_shorthand,
        division,
        conference
    FROM {S}.silver_teams
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_player AS
    SELECT
        MD5(CAST(player_id AS STRING)) AS player_sk,
        player_id,
        first_name,
        last_name,
        CONCAT(first_name, ' ', last_name) AS full_name,
        position,
        role,
        status,
        current_team_id
    FROM {S}.silver_players
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_venue AS
    SELECT
        MD5(CAST(venue_id AS STRING)) AS venue_sk,
        venue_id, venue_name, city, state, country, timezone, capacity
    FROM {S}.silver_venues
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_game AS
    SELECT
        MD5(CAST(g.game_id AS STRING)) AS game_sk,
        g.game_id,
        g.season,
        g.stage,
        g.game_date,
        g.scheduled_time,
        g.venue_id,
        MD5(CAST(g.venue_id AS STRING)) AS venue_sk,
        g.home_team_id,
        MD5(CAST(g.home_team_id AS STRING)) AS home_team_sk,
        g.away_team_id,
        MD5(CAST(g.away_team_id AS STRING)) AS away_team_sk,
        g.winning_team_id,
        ht.team_shorthand AS home_team_shorthand,
        at.team_shorthand AS away_team_shorthand
    FROM {S}.silver_games g
    LEFT JOIN {S}.silver_teams ht ON ht.team_id = g.home_team_id
    LEFT JOIN {S}.silver_teams at ON at.team_id = g.away_team_id
""")

# A simple date dimension covering 2020-01-01 .. today + 90d
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.dim_date AS
    SELECT
        DATE_FORMAT(d, 'yyyyMMdd') AS date_sk,
        d AS date,
        YEAR(d) AS year,
        MONTH(d) AS month,
        DAY(d) AS day,
        DATE_FORMAT(d, 'EEEE') AS day_name,
        WEEKOFYEAR(d) AS week_of_year,
        QUARTER(d) AS quarter
    FROM (
        SELECT EXPLODE(SEQUENCE(DATE'2020-01-01', DATE_ADD(CURRENT_DATE(), 90), INTERVAL 1 DAY)) AS d
    )
""")
print("dimensions built.")

dimensions built.


## Facts

In [5]:
# fact_game_events — joins event grain to game + team for analytics
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_game_events (
        event_sk         STRING NOT NULL,
        game_sk          STRING,
        game_id          INT,
        season           STRING,
        game_date        DATE,
        period           INT,
        period_time      INT,
        x_coord          INT,
        y_coord          INT,
        zone             STRING,
        team             STRING,
        team_id          INT,
        team_sk          STRING,
        player_id        INT,
        player_sk        STRING,
        event_name       STRING,
        event_type       STRING,
        outcome          STRING,
        is_shot          BOOLEAN,
        is_goal          BOOLEAN
    )
    CLUSTER BY (season, game_date, team_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.fact_game_events
    SELECT
        e.base_event_id              AS event_sk,
        MD5(CAST(e.game_id AS STRING)) AS game_sk,
        e.game_id,
        g.season,
        g.game_date,
        e.period,
        e.period_time,
        e.x_coord,
        e.y_coord,
        e.zone,
        e.team,
        CASE WHEN UPPER(e.team) = UPPER(ht.team_shorthand) THEN g.home_team_id
             WHEN UPPER(e.team) = UPPER(at.team_shorthand) THEN g.away_team_id
             ELSE NULL END           AS team_id,
        MD5(CAST(CASE WHEN UPPER(e.team) = UPPER(ht.team_shorthand) THEN g.home_team_id
                      WHEN UPPER(e.team) = UPPER(at.team_shorthand) THEN g.away_team_id END AS STRING)) AS team_sk,
        e.player_id,
        MD5(CAST(e.player_id AS STRING)) AS player_sk,
        e.name                       AS event_name,
        e.type                       AS event_type,
        e.outcome                    AS outcome,
        LOWER(e.name) LIKE '%shot%' OR LOWER(e.shorthand) LIKE '%shot%' AS is_shot,
        LOWER(e.outcome) = 'goal'    AS is_goal
    FROM {S}.silver_compiled_events e
    JOIN {S}.silver_games g ON g.game_id = e.game_id
    LEFT JOIN {S}.silver_teams ht ON ht.team_id = g.home_team_id
    LEFT JOIN {S}.silver_teams at ON at.team_id = g.away_team_id
""")
print("fact_game_events:", spark.table(f"{G}.fact_game_events").count())

fact_game_events: 150


In [6]:
# fact_player_shifts — joins shift events to game and team context
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_player_shifts (
        shift_sk     STRING NOT NULL,
        game_sk      STRING,
        game_id      INT,
        player_sk    STRING,
        player_id    INT,
        period       INT,
        period_time  INT,
        team         STRING,
        type         STRING,
        outcome      STRING,
        x_coord      INT,
        y_coord      INT,
        zone         STRING
    )
    CLUSTER BY (game_id, player_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.fact_player_shifts
    SELECT
        s.shift_event_sk AS shift_sk,
        MD5(CAST(s.game_id AS STRING)) AS game_sk,
        s.game_id,
        MD5(CAST(s.player_id AS STRING)) AS player_sk,
        s.player_id,
        s.period, s.period_time,
        s.team,  s.type, s.outcome,
        s.x_coord, s.y_coord, s.zone
    FROM {S}.silver_shift_events s
""")
print("fact_player_shifts:", spark.table(f"{G}.fact_player_shifts").count())

fact_player_shifts: 75


In [7]:
# fact_player_game_metrics — pivoted view kept long-form for flexibility (pivot in BI)
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_player_game_metrics (
        metric_sk    STRING NOT NULL,
        game_sk      STRING,
        game_id      INT,
        player_sk    STRING,
        player_id    INT,
        team_sk      STRING,
        team_id      INT,
        period       INT,
        position     STRING,
        topic_id     STRING,
        metric_key   STRING,
        metric_value DOUBLE
    )
    CLUSTER BY (game_id, player_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.fact_player_game_metrics
    SELECT
        m.metric_sk,
        MD5(CAST(m.game_id   AS STRING)) AS game_sk,
        m.game_id,
        MD5(CAST(m.player_id AS STRING)) AS player_sk,
        m.player_id,
        MD5(CAST(m.team_id   AS STRING)) AS team_sk,
        m.team_id,
        m.period,
        m.position,
        m.topic_id,
        m.metric_key,
        m.metric_value
    FROM {S}.silver_player_game_metrics m
""")
print("fact_player_game_metrics:", spark.table(f"{G}.fact_player_game_metrics").count())

fact_player_game_metrics: 720


In [8]:
# fact_player_season_metrics
spark.sql(f"""
    CREATE OR REPLACE TABLE {G}.fact_player_season_metrics (
        season_metric_sk STRING NOT NULL,
        scope            STRING,
        season           STRING,
        topic_id         STRING,
        player_sk        STRING,
        player_id        INT,
        team_sk          STRING,
        team_id          INT,
        position         STRING,
        metric_key       STRING,
        metric_value     DOUBLE
    )
    CLUSTER BY (season, player_id)
""")
spark.sql(f"""
    INSERT OVERWRITE {G}.fact_player_season_metrics
    SELECT
        m.season_metric_sk,
        m.scope,
        m.season,
        m.topic_id,
        MD5(CAST(m.player_id AS STRING)) AS player_sk,
        m.player_id,
        MD5(CAST(m.team_id   AS STRING)) AS team_sk,
        m.team_id,
        m.position,
        m.metric_key,
        m.metric_value
    FROM {S}.silver_season_metrics m
""")
print("fact_player_season_metrics:", spark.table(f"{G}.fact_player_season_metrics").count())

fact_player_season_metrics: 696


## Analytical views

In [9]:
# Standings (joins team records + team for shorthand/conference/division)
spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_team_standings AS
    SELECT
        t.team_shorthand,
        t.team_name,
        t.conference,
        t.division,
        r.wins,
        r.losses,
        r.overtime_losses,
        r.standings_points,
        ROUND(r.standings_points * 1.0 / NULLIF(r.wins + r.losses + r.overtime_losses, 0), 3) AS points_per_game
    FROM {UC_CATALOG}.{SILVER_SCHEMA}.silver_team_records r
    JOIN {UC_CATALOG}.{SILVER_SCHEMA}.silver_teams t USING (team_id)
""")

# Player season leaderboard for any single metric_key (Genie + dashboard friendly)
spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_player_season_leaders AS
    SELECT
        m.scope, m.season, m.topic_id, m.metric_key,
        p.full_name, p.position,
        t.team_shorthand,
        m.metric_value
    FROM {G}.fact_player_season_metrics m
    LEFT JOIN {G}.dim_player p USING (player_id)
    LEFT JOIN {G}.dim_team   t USING (team_id)
    WHERE m.scope = 'player'
""")

# Shot map — pre-aggregated for plotting
spark.sql(f"""
    CREATE OR REPLACE VIEW {G}.v_shot_map AS
    SELECT
        season, team, x_coord, y_coord, zone,
        SUM(CASE WHEN is_goal THEN 1 ELSE 0 END) AS goals,
        COUNT(*) AS shot_attempts
    FROM {G}.fact_game_events
    WHERE is_shot AND x_coord IS NOT NULL AND y_coord IS NOT NULL
    GROUP BY season, team, x_coord, y_coord, zone
""")

print("views built.")

views built.


## Verify

In [10]:
print("=" * 60)
print("GOLD LAYER SUMMARY")
print("=" * 60)
for t in ["dim_team","dim_player","dim_venue","dim_game","dim_date",
          "fact_game_events","fact_player_shifts",
          "fact_player_game_metrics","fact_player_season_metrics"]:
    full = f"{G}.{t}"
    n = spark.table(full).count()
    print(f"  {full:<70s}  {n:>10,} rows")
print()
spark.sql(f"SELECT * FROM {G}.v_team_standings ORDER BY standings_points DESC LIMIT 8").show(truncate=False)

GOLD LAYER SUMMARY


  alexander_booth.sportlogiq_nhl_gold.dim_team                                     8 rows


  alexander_booth.sportlogiq_nhl_gold.dim_player                                  50 rows


  alexander_booth.sportlogiq_nhl_gold.dim_venue                                    8 rows


  alexander_booth.sportlogiq_nhl_gold.dim_game                                     5 rows


...

  alexander_booth.sportlogiq_nhl_gold.fact_game_events                           150 rows


  alexander_booth.sportlogiq_nhl_gold.fact_player_shifts                          75 rows


  alexander_booth.sportlogiq_nhl_gold.fact_player_game_metrics                   720 rows


  alexander_booth.sportlogiq_nhl_gold.fact_player_season_metrics                 696 rows



+--------------+--------------------+----------+------------+----+------+---------------+----------------+---------------+
|team_shorthand|team_name           |conference|division    |wins|losses|overtime_losses|standings_points|points_per_game|
+--------------+--------------------+----------+------------+----+------+---------------+----------------+---------------+
|VGK           |Vegas Golden Knights|Western   |Pacific     |37  |23    |7              |81              |1.209          |
|NYR           |New York Rangers    |Eastern   |Metropolitan|35  |27    |7              |77              |1.116          |
|CHI           |Chicago Blackhawks  |Western   |Central     |34  |19    |4              |72              |1.263          |
|TBL           |Tampa Bay Lightning |Eastern   |Atlantic    |30  |21    |7              |67              |1.155          |
|BUF           |Buffalo Sabres      |Eastern   |Atlantic    |30  |27    |4              |64              |1.049          |
|BOS           |